# Enriching Dimensionality Reduction with Distortion Cues
---

This notebook reproduces the main analysis presented in the paper *Enriching Dimensionality Reduction with Distortion Cues*. The workflow computes a two-dimensional projection, constructs a Delaunay triangulation, estimates projection distortions through high-dimensional edge interpolation, and generates the data for visualizations used throughout the paper.

## 1. Import Dependencies

Import the required libraries together with the implementation of the proposed distortion cue framework.

In [26]:
%reset -f

In [27]:
from distortion_cues import ProjectionAnalysisVisualizer, visualizer, utility
from datasets.datasets import *

from distortion_cues import config as cfg

## 2. Dataset Configuration

Select the dataset and configure the parameters required for the projection analysis.

The framework supports both synthetic datasets, used for controlled experiments, and real-world datasets for practical evaluation.

In [28]:
# dataset = 'cube_same_size_same_dist' # "cube_diff_size_diff_dist", "cube_diff_size_same_dist", "har", "aedessex", "mnsit"

dataset = 'cube_diff_size_diff_dist'    # Choose dataset from above list that you need to analyse. Also, you can include your own dataset too.
method = 'tsne'   #tsne, Umap

In [29]:
np.random.seed(cfg.GLOBAL_SEED)

# Synthetic dataset 
num_dim = cfg.NUM_DIMENSION
n_pts_per_gauss = cfg.POINTS_PER_CUBE

runtime_repetition = cfg.RUNTIME_REPETITION

cfg_dt = cfg.get_config(dataset, method)
perplexity = cfg_dt["perplexity"]

## 3. Output Configuration

Configure the directory used to store the generated embeddings, intermediate results, distortion cues, and figures.

In [30]:
save_output = True

In [31]:
if save_output:
    output_folder = f"results/{dataset}/{method}_perp_{perplexity}_plots"
    os.makedirs(output_folder, exist_ok=True)
else:
    output_folder = None

## 4. Load the Dataset

Load the selected high-dimensional dataset together with the associated labels required for visualization and quantitative evaluation.

In [ ]:
# For synthetic data only
cube_sizes, cube_centers = utility.cubic_centers_sizes(dataset=dataset)

In [33]:

D,c, dim, output_size, n_gauss = selected_dataset_dt(dataset, num_dim, n_pts_per_gauss, cube_sizes=cube_sizes, cube_centers=cube_centers, cluster_spacing = 1.0, spread_factor = 0.01)
class_label = c.astype(int)

class_label = class_label - class_label.min()
np.unique(class_label)

array([0, 1, 2, 3])

In [34]:
np.save(f"{output_folder}/D_{dataset}.npy", D)
np.save(f"{output_folder}/class_label_{dataset}.npy", class_label)


## 5. Initialize the Projection Analysis Framework

Initialize the proposed framework using the high-dimensional data and the selected dimensionality reduction technique.

Lets initialize the framework using original data and employing *TSNE* as a projection method. However, you can use other projections also.

In [35]:
proj_viz = ProjectionAnalysisVisualizer(data = D, class_label= class_label, projection_method= method, perplexity=perplexity, output_path=output_folder)  # Supervised

## 6. Compute the Low-Dimensional Projection

Generate the two-dimensional embedding that serves as the basis for the subsequent distortion analysis.

In [36]:
low_dm_emb = proj_viz.projection_emb_low_dim()

In [37]:
np.save(f"{output_folder}/low_dm_emb_{dataset}.npy", low_dm_emb)


## 7. High-Dimensional Distance Analysis

Compute inter- and intra-cluster distance statistics in the original high-dimensional space. These measurements provide a quantitative reference for evaluating relationships preserved in the low-dimensional projection.

In [38]:
(distance_matrix_hd_true, mean_cluster_distance__hd_true), mean_time_pair_dist, std_time_pair_dist = utility.run_and_measure(
                                                    proj_viz.inter_intra_cluster_pairwise_distance,
                                                    D,
                                                    class_label,
                                                    repeats=runtime_repetition
                                                )

print(f"Pairwise Distance Matrix runtime: {mean_time_pair_dist:.4f} ± {std_time_pair_dist:.4f} sec")

Pairwise Distance Matrix runtime: 0.0155 ± 0.0000 sec


In [39]:
np.save(f"{output_folder}/distance_matrix_hd_true_{dataset}.npy", distance_matrix_hd_true)
np.save(f"{output_folder}/mean_cluster_distance__hd_true_{dataset}.npy", mean_cluster_distance__hd_true)



## 8. Construct the Delaunay Triangulation

Construct a Delaunay triangulation over the projected embedding. The triangulation defines the neighborhood structure used for high-dimensional edge interpolation.

In [40]:
# tri_delaunay = proj_viz.delaunay_triangulation(embedding= low_dm_emb)

Use the following code if you do not want to measure runtime otherwise uncomment above and use directly.

In [41]:
tri_delaunay, mean_time, std_time = utility.run_and_measure(
    proj_viz.delaunay_triangulation,
    embedding=low_dm_emb,
    repeats=runtime_repetition
)

print(f"Delaunay runtime: {mean_time:.6f} ± {std_time:.6f} sec")

Delaunay runtime: 0.004890 ± 0.000000 sec


In [42]:
np.savez(f"{output_folder}/tri_delaunay_{dataset}.npz", simplices=tri_delaunay.simplices)


In [43]:
edges_delaunay,_ = utility.extract_delaunay_edges_2d(tri_delaunay)
np.save(
        f"{output_folder}/edges_delaunay_orig_{dataset}.npy",
        edges_delaunay
    )

### High-Dimensional Edge Interpolation

For each Delaunay edge, the corresponding relationship in the original high-dimensional space is estimated through interpolation. These estimates provide the basis for computing the distortion cues introduced in the paper.

## 9. Compute Distortion Cues

Estimate projection distortions by interpolating distances along Delaunay edges and comparing them with the corresponding relationships in the original high-dimensional space.

The resulting distortion values are used to enrich the projection with geometric cues that facilitate the interpretation of projection quality.

In [44]:
intensity_interp_cordinates, max_interp, min_interp  = proj_viz.delanay_hd_edge_lenghts_inter()


uncomment below to measure runtime

In [45]:
# intensity_interp_cordinates, min_interp, max_interp, mean_time_delaunay, std_time_delaunay = utility.run_and_measure(
#     proj_viz.delanay_hd_edge_lenghts_inter,
#     repeats=runtime_repetition
# )

# print(f"Delaunay plot runtime: {mean_time_delaunay:.4f} ± {std_time_delaunay:.4f} sec")





In [46]:
np.save(f"{output_folder}/intensity_interp_cordinates_{dataset}.npy", intensity_interp_cordinates)


## 10. Quantitative Evaluation (CheckViz)

Evaluate the projection using established distortion measures, including Normalized Local Metric (NLM) and Curvilinear Component Analysis (CCA), to complement the proposed distortion cue analysis.

In [47]:
# nlm, cca = proj_viz.compute_distortion_prob(D, low_dm_emb) 

In [48]:
(nlm, cca), mean_time_check_dist_prob, std_time_check_dist_prob = utility.run_and_measure(
    proj_viz.compute_distortion_prob,
    D,
    low_dm_emb,
    repeats=runtime_repetition
)

print(f"CheckViz plot runtime: {mean_time_check_dist_prob:.4f} ± {std_time_check_dist_prob:.4f} sec")


CheckViz plot runtime: 0.1744 ± 0.0000 sec


In [49]:
np.save(f"{output_folder}/nlm_{dataset}.npy", np.asarray(nlm))
np.save(f"{output_folder}/cca_{dataset}.npy", np.asarray(cca))

In [50]:
# _, mean_time_check, std_time_check = utility.run_and_measure(
#     visualizer.plot_checkviz,
#     low_dm_emb,
#     cca,
#     nlm,
#     class_label=class_label,
#     colormap_clust = "Plasma",
#     filename="checkvis_cca_nlm",
#     output_path=output_folder,
#     dpi=cfg.DPI,
#     repeats=runtime_repetition
# )

# print(f"CheckViz plot runtime: {mean_time_check:.4f} ± {std_time_check:.4f} sec")

# mean_time_check_final = mean_time_check_dist_prob + mean_time_check
# std_time_check_final = std_time_check_dist_prob + std_time_check

# print(f"CheckViz plot Final runtime: {mean_time_check_final:.4f} ± {std_time_check_final:.4f} sec")
